## README:

* Cells labeled with '## ----- CONFIG ----- ##' contain parameters that need to be set manually 

In [1]:
import io
import os
import pandas as pd

from contextlib import redirect_stdout
from dotenv import load_dotenv

## Global config

In [2]:
## ----- CONFIG ----- ##
WRITE_MODE = 'w' # 'x': do not overwrite

## Load env variables

In [3]:
load_dotenv(r'../.env')

True

In [5]:
DATA_DIR = os.getenv('DATA_DIR')

# create data directory if does not exist
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, 'FRED'), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, 'YFINANCE'), exist_ok=True)

## FRED API - Economic data

In [6]:
import pyfredapi as pf
from pyfredapi._base import FredAPIRequestError 

API_KEY = os.getenv('FRED_API_KEY')

### Configuration

In [7]:
## ----- CONFIG ----- ##
VERSION = '001'

FRED_PARAMS = {
    'observation_start': '1975-01-01',
    'observation_end': '2025-12-05'
}

SERIES_ID_LIST = [
    'VIXCLS',
    'DFF',
    'T10Y2Y',
    'REAINTRATREARAT10Y',
    'BAMLH0A0HYM2',
    'CPIAUCSL',
    'ICSA',
    'PPIACO',
    'USSTHPI',
]

### Collect data

In [8]:
# store printed string
buffer = io.StringIO()

# download data for each series
with redirect_stdout(buffer):
    for series_id in SERIES_ID_LIST:
        # get series info
        try:
            series_info = pf.get_series_info(series_id=series_id, api_key=API_KEY)
        except FredAPIRequestError as e:
            msg = str(e)
            if 'The series does not exist' in msg:
                print(f'The series "{series_id}" does not exist')
            else:
                print(msg)
            continue

        # print key series info.
        print(f'id: {series_id} | title: {series_info.title}')
        print(f'obs range: {series_info.observation_start} to {series_info.observation_end}')
        print(f'freq: {series_info.frequency} | units: {series_info.units}')
        print(series_info.seasonal_adjustment)

        # get data
        df = pf.get_series(series_id=series_id, api_key=API_KEY, **FRED_PARAMS)
        print(df.shape, '\n')

        # export data to local storage
        # skips if already exists (prevents overwrite)
        file_name = f'{series_id}_{VERSION}.csv'
        out_path = os.path.join(DATA_DIR, 'FRED', file_name)
        try:
            df.to_csv(out_path, index=False, mode=WRITE_MODE)
        except FileExistsError:
            print(f'{file_name} already exists in the output directory.')

In [9]:
# write metadata
metadata_file = os.path.join(DATA_DIR, 'FRED', f"metadata_{VERSION}.txt")
with open(metadata_file, "w") as f:
    f.write(buffer.getvalue())

## yfinance - Stock prices

In [10]:
import yfinance as yf

### Configuration

In [11]:
## ----- CONFIG ----- ##
YF_VERSION = '001'

YF_PARAMS = {
    'period': 'max',
    'interval': '1d',
    'start': '1975-01-01',
    'end': '2025-12-05',
    'keepna': True,
    'actions': False,
    'multi_level_index': True
}

TICKER_LIST = [
    'SPY', 
    'AAPL', 
    'NVDA', 
    'MSFT', 
    'AMZN', 
    'GOOG', 
    'JPM', 
    'XOM',
    'PG', 
    'UNH'
]

### Collect data

In [12]:
df = yf.download(
    tickers=TICKER_LIST,
    **YF_PARAMS
)

df.shape

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  10 of 10 completed


(12840, 50)

In [13]:
# export data to local storage
# skips if already exists (prevents overwrite)
file_name = f"prices_{YF_VERSION}.csv"
out_path = os.path.join(DATA_DIR, 'YFINANCE', file_name)
try:
    df.reset_index().to_csv(out_path, index=False, mode=WRITE_MODE)
except FileExistsError:
    print(f"{file_name} already exists in the output directory.")